In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import joblib
import numpy as np

# Load dataset (replace 'Flight_Delay.csv' with actual file name)
df = pd.read_csv('Flight_Delay.csv')

# Basic preprocessing
# Drop unnecessary columns or those with too many missing values
df = df.drop(['Date', 'TailNum', 'CancellationCode', 'Cancelled', 'Diverted'], axis=1, errors='ignore')  # Adjust if needed

# Handle missing values
df = df.dropna(subset=['ArrDelay'])  # Target can't be null
df.fillna(0, inplace=True)  # Fill delays with 0

# Encode categorical columns
label_encoders = {}
for col in ['UniqueCarrier', 'Airline', 'Origin', 'Org_Airport', 'Dest', 'Dest_Airport']:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

# Features and target (predict ArrDelay - delay in minutes)
X = df.drop(['ArrDelay'], axis=1)  # All columns except target
y = df['ArrDelay']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor (for good accuracy)
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# Predictions and accuracy
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Mean Absolute Error (MAE): {mae}')  # Should be low, e.g., \~10-20 min
print(f'Root Mean Squared Error (RMSE): {rmse}')  # Good if <40-50

# Save model and encoders
joblib.dump(model, 'flight_delay_model.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')
print("Model saved successfully!")

Mean Absolute Error (MAE): 3.3126420974731445
Root Mean Squared Error (RMSE): 9.010358147991573
Model saved successfully!
